# Short Fold-Local Drawdown-Drift Retention Audit

Purpose: aggregate the current strict retention gate, SOXLBacktest Qlib reference, short-volshock official replay, and short PPO screen evidence into one executable model-selection audit. The audit decides whether any current candidate can be retained as an actor under the requested walk-forward / drawdown / return-alignment constraints.

In [1]:

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'SPEC.md').exists():
            return candidate
    raise FileNotFoundError('could not locate repository root from notebook cwd')


ROOT = find_repo_root(Path.cwd())
CARD = ROOT / 'runtime/model_selection/card/short_foldlocal_drawdown_drift_retention_audit_v1'
CARD.mkdir(parents=True, exist_ok=True)

EXPERIMENT = 'short_foldlocal_drawdown_drift_retention_audit_v1'
SOURCE_PATHS = {
    'final_handoff_summary': 'runtime/model_selection/card/final_walkforward_retraining_handoff_v1/summary.json',
    'final_handoff_selection': 'runtime/model_selection/card/final_walkforward_retraining_handoff_v1/selection.json',
    'strict_walkforward_gate': 'runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json',
    'strict_candidate_filter': 'runtime/model_selection/card/strict_candidate_retention_filter_v1/summary.json',
    'soxlbacktest_qlib_summary': 'runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1/summary.json',
    'soxlbacktest_qlib_gate_summary': 'runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1/candidate_gate_summary.csv',
    'soxlbacktest_reference_score': 'runtime/model_selection/card/soxlbacktest_reference_model_score_v1/summary.json',
    'short_reward_summary': 'runtime/reward/card/short_volshock_drawdown_drift_reward_short_ppo_screen_v1/summary.json',
    'short_reward_selection': 'runtime/reward/card/short_volshock_drawdown_drift_reward_short_ppo_screen_v1/selection.json',
    'short_reward_checkpoint_scores': 'runtime/reward/card/short_volshock_drawdown_drift_reward_short_ppo_screen_v1/constrained_checkpoint_scores.csv',
    'official_replay_summary': 'runtime/model_selection/card/short_volshock_official_replay_screen_v1/summary.json',
    'official_replay_candidates': 'runtime/model_selection/card/short_volshock_official_replay_screen_v1/official_replay_candidate_summary.csv',
    'sell_guard_summary': 'runtime/model_selection/card/short_volshock_sell_guard_official_replay_v1/summary.json',
    'sell_guard_candidates': 'runtime/model_selection/card/short_volshock_sell_guard_official_replay_v1/official_replay_candidate_summary.csv',
    'bc_checkpoint_alignment_summary': 'runtime/model_selection/card/short_volshock_bc_checkpoint_alignment_v1/summary.json',
    'bc_checkpoint_alignment_scores': 'runtime/model_selection/card/short_volshock_bc_checkpoint_alignment_v1/checkpoint_objective_scores.csv',
    'cap10_readiness_summary': 'runtime/model_selection/card/cap10_foldlocal_multi_objective_readiness_v1/summary.json',
}


def repo_path(rel: str) -> Path:
    return ROOT / rel


def load_json(rel: str) -> dict:
    return json.loads(repo_path(rel).read_text(encoding='utf-8'))


def load_csv(rel: str) -> pd.DataFrame:
    return pd.read_csv(repo_path(rel))


def fmt(value) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    try:
        return f'{float(value):.6f}'
    except (TypeError, ValueError):
        return str(value)


def bool_str(value: bool) -> str:
    return 'True' if bool(value) else 'False'

source_manifest = pd.DataFrame(
    [
        {
            'source_label': label,
            'path': rel,
            'exists': repo_path(rel).exists(),
            'artifact_type': Path(rel).suffix.lstrip('.') or 'directory',
        }
        for label, rel in SOURCE_PATHS.items()
    ]
)
if not bool(source_manifest['exists'].all()):
    missing = source_manifest.loc[~source_manifest['exists'], 'path'].tolist()
    raise FileNotFoundError(f'missing source artifacts: {missing}')

final_summary = load_json(SOURCE_PATHS['final_handoff_summary'])
final_selection = load_json(SOURCE_PATHS['final_handoff_selection'])
strict_gate_payload = load_json(SOURCE_PATHS['strict_walkforward_gate'])
strict_gate = strict_gate_payload['summary']
strict_filter_payload = load_json(SOURCE_PATHS['strict_candidate_filter'])
strict_filter = strict_filter_payload['summary']
soxl_summary = load_json(SOURCE_PATHS['soxlbacktest_qlib_summary'])
short_reward_summary = load_json(SOURCE_PATHS['short_reward_summary'])
short_reward_selection = load_json(SOURCE_PATHS['short_reward_selection'])
official_summary = load_json(SOURCE_PATHS['official_replay_summary'])
sell_guard_summary = load_json(SOURCE_PATHS['sell_guard_summary'])
bc_alignment_summary = load_json(SOURCE_PATHS['bc_checkpoint_alignment_summary'])
cap10_summary = load_json(SOURCE_PATHS['cap10_readiness_summary'])

soxl_gate = load_csv(SOURCE_PATHS['soxlbacktest_qlib_gate_summary'])
reward_scores = load_csv(SOURCE_PATHS['short_reward_checkpoint_scores'])
official_candidates = load_csv(SOURCE_PATHS['official_replay_candidates'])
sell_guard_candidates = load_csv(SOURCE_PATHS['sell_guard_candidates'])
bc_scores = load_csv(SOURCE_PATHS['bc_checkpoint_alignment_scores'])

rows: list[dict[str, str]] = []

def add_row(**kwargs) -> None:
    defaults = {
        'source_label': '',
        'candidate_id': '',
        'candidate_type': '',
        'windows_evaluated': '',
        'wf2020_covered': 'False',
        'min_relative_return_pp': '',
        'max_dd_pct': '',
        'ytd_return_pct': '',
        'walkforward_drawdown_pass': 'False',
        'current_best_alignment_pass': 'False',
        'screen_gate_pass': 'False',
        'retention_gate_pass': 'False',
        'promotion_allowed': 'False',
        'actor_retention_allowed': 'False',
        'blocker': '',
        'next_mechanism': '',
    }
    defaults.update({key: str(value) for key, value in kwargs.items()})
    rows.append(defaults)

add_row(
    source_label='model_retention_walkforward_gate_v1',
    candidate_id='strict_profile_pool',
    candidate_type='profile_pool',
    windows_evaluated=','.join(strict_gate.get('required_wf_windows') or []),
    wf2020_covered='True',
    walkforward_drawdown_pass=bool_str((strict_gate.get('passing_profile_count') or 0) > 0),
    current_best_alignment_pass=bool_str((strict_gate.get('passing_profile_count') or 0) > 0),
    screen_gate_pass=bool_str((strict_gate.get('passing_profile_count') or 0) > 0),
    blocker='strict_retention_gate_has_zero_passing_profiles',
    next_mechanism=str(strict_filter.get('next_mechanism')),
)

for candidate in ['legacy_default', 'wf_pass_ytd18_15_15_02']:
    row = soxl_gate.loc[soxl_gate['candidate'].eq(candidate)].iloc[0]
    failures = []
    if not bool(row['wf_drawdown_pass']):
        failures.append('wf_max_drawdown_ge_20')
    if not bool(row['ytd_soxlbacktest_return_pass']):
        failures.append('return_below_soxlbacktest_reference')
    if not bool(row['ytd_current_best_alignment_pass']):
        failures.append('return_below_current_best_alignment')
    add_row(
        source_label='soxlbacktest_qlib_walkforward_baseline_v1',
        candidate_id=f'soxlbacktest::{candidate}',
        candidate_type='qlib_reference_screen',
        windows_evaluated=str(row['wf_windows_present']),
        wf2020_covered='True',
        min_relative_return_pp=fmt(row['min_wf_return_pct']),
        max_dd_pct=fmt(row['worst_wf_max_drawdown_pct']),
        ytd_return_pct=fmt(row['ytd_return_pct']),
        walkforward_drawdown_pass=bool_str(row['wf_drawdown_pass']),
        current_best_alignment_pass=bool_str(row['ytd_current_best_alignment_pass']),
        screen_gate_pass=bool_str(row['retention_gate_pass']),
        blocker=';'.join(failures),
        next_mechanism='score_trained_candidates_against_soxlbacktest_reference',
    )

for _, row in reward_scores.iterrows():
    candidate = str(row['candidate'])
    if candidate == 'bc_initialization':
        blocker = 'missing_wf_2020;short_training_screen_only_no_actor_retention'
    elif candidate == 'reward_ppo_selected':
        blocker = f'ppo_drawdown_drift:wf_2022_max_dd_{fmt(row["max_max_dd"])}'
    else:
        blocker = 'short_training_screen_only_no_actor_retention'
    add_row(
        source_label='short_volshock_drawdown_drift_reward_short_ppo_screen_v1',
        candidate_id=f'short_reward::{candidate}',
        candidate_type='short_training_screen',
        windows_evaluated='wf_2022,wf_2023_2024',
        wf2020_covered='False',
        min_relative_return_pp=fmt(row['min_relative_return']),
        max_dd_pct=fmt(row['max_max_dd']),
        walkforward_drawdown_pass=bool_str(float(row['max_max_dd']) <= float(row['target_max_dd_pct'])),
        current_best_alignment_pass='False',
        screen_gate_pass=bool_str(row['selector_gate_pass']),
        blocker=blocker,
        next_mechanism=str(short_reward_selection.get('next_mechanism')),
    )

for variant, source_label, frame in [
    ('short_volshock_cap05_hit2', 'short_volshock_official_replay_screen_v1', official_candidates),
    ('short_volshock_cap05_hit2_no_guard', 'short_volshock_sell_guard_official_replay_v1', sell_guard_candidates),
]:
    row = frame.loc[frame['variant'].eq(variant)].iloc[0]
    add_row(
        source_label=source_label,
        candidate_id=f'official_replay::{variant}',
        candidate_type='official_warm_start_replay',
        windows_evaluated='wf_2022,wf_2023_2024',
        wf2020_covered='False',
        min_relative_return_pp=fmt(row['min_relative_return']),
        max_dd_pct=fmt(row['max_max_dd']),
        walkforward_drawdown_pass=bool_str(float(row['max_max_dd']) < 20.0),
        current_best_alignment_pass='False',
        screen_gate_pass=bool_str(row['full_gate_all']),
        blocker='near_miss_no_promotion:missing_wf_2020_or_30pp_15dd_gate',
        next_mechanism='checkpoint_objective_or_fold_local_drawdown_drift_training_target',
    )

bc_row = bc_scores.loc[bc_scores['candidate'].eq('short_volshock_plain')].iloc[0]
add_row(
    source_label='short_volshock_bc_checkpoint_alignment_v1',
    candidate_id='bc_alignment::short_volshock_plain',
    candidate_type='bc_checkpoint_alignment',
    windows_evaluated='wf_2022,wf_2023_2024',
    wf2020_covered='False',
    min_relative_return_pp=fmt(bc_row['min_relative_return']),
    max_dd_pct=fmt(bc_row['max_max_dd']),
    walkforward_drawdown_pass=bool_str(float(bc_row['max_max_dd']) < 20.0),
    current_best_alignment_pass='False',
    screen_gate_pass=bool_str(bc_row['window_gate']),
    blocker='official_warm_start_replay_gate_failed_no_promotion',
    next_mechanism=str(bc_alignment_summary.get('next_mechanism')),
)

add_row(
    source_label='cap10_foldlocal_multi_objective_readiness_v1',
    candidate_id='cap10_foldlocal_multiobjective',
    candidate_type='fold_local_target_readiness',
    windows_evaluated='wf_2022_train,wf_2023_2024_train',
    wf2020_covered='False',
    blocker='target_readiness_not_training_ready_as_is',
    next_mechanism=str(cap10_summary.get('next_mechanism')),
)

candidate_matrix = pd.DataFrame(rows)
retention_gate_pass_count = int(candidate_matrix['retention_gate_pass'].eq('True').sum())

best_near_miss = candidate_matrix.loc[
    candidate_matrix['candidate_id'].eq('official_replay::short_volshock_cap05_hit2_no_guard')
].iloc[0].to_dict()

required_gates = {
    'required_wf_windows': list(strict_gate.get('required_wf_windows') or []),
    'max_walkforward_drawdown_pct': float(strict_gate.get('max_walkforward_drawdown_pct')),
    'required_best_return_pct': float(strict_gate.get('required_best_return_pct')),
    'current_best_return_pct': float(strict_gate.get('current_best_return_pct')),
    'reference_profile': strict_gate.get('reference_profile'),
    'reference_return_pct': float(strict_gate.get('reference_return_pct')),
    'reference_max_drawdown_pct': float(strict_gate.get('reference_max_drawdown_pct')),
    'short_screen_target_relative_return_pp': 30.0,
    'short_screen_target_max_dd_pct': 15.0,
}

summary = {
    'experiment': EXPERIMENT,
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'decision': 'short_foldlocal_drawdown_drift_no_retained_actor',
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'retention_gate_pass_count': retention_gate_pass_count,
    'candidate_row_count': int(len(candidate_matrix)),
    'source_card_count': int(len(source_manifest)),
    'required_gates': required_gates,
    'strict_candidate_retention': {
        'decision': strict_filter.get('decision'),
        'passing_profile_count': int(strict_filter.get('passing_profile_count') or 0),
        'promotable_profile_count': int(strict_filter.get('promotable_profile_count') or 0),
        'actor_retention_allowed_count': int(strict_filter.get('actor_retention_allowed_count') or 0),
        'reference_visible_count': int(strict_filter.get('reference_visible_count') or 0),
        'ppo_allowed_next': bool(strict_filter.get('ppo_allowed_next')),
        'next_mechanism': strict_filter.get('next_mechanism'),
    },
    'soxlbacktest_reference': {
        'profile': 'soxl_backtest_policy_reimplementation_v1',
        'role': 'qlib_reference_baseline_not_actor',
        'summary_decision': soxl_summary.get('decision'),
        'promotion_allowed': bool(soxl_summary.get('promotion_allowed')),
        'actor_retention_allowed': bool(soxl_summary.get('actor_retention_allowed')),
    },
    'short_reward_screen': {
        'decision': short_reward_summary.get('decision'),
        'promotion_allowed': bool(short_reward_summary.get('promotion_allowed')),
        'bc_initialization_selector_gate_pass': True,
        'reward_ppo_selected_max_dd_pct': float(
            reward_scores.loc[reward_scores['candidate'].eq('reward_ppo_selected'), 'max_max_dd'].iloc[0]
        ),
    },
    'best_near_miss': best_near_miss,
    'conclusion': [
        'No current trained actor or short training candidate is eligible for retention.',
        'SOXLBacktest is preserved as a Qlib-aligned buy/sell reference baseline, not as a runnable actor.',
        'BC initialization is useful diagnostic evidence on two evaluated folds but cannot be retained because wf_2020 and full fixed-split validation are missing.',
        'The reward PPO selected checkpoint expands drawdown in wf_2022, so longer PPO is blocked until drawdown-drift is part of the objective/checkpoint gate.',
        'The closest official replay remains a near miss, not promotion evidence.',
    ],
}

selection = {
    'experiment': EXPERIMENT,
    'decision': summary['decision'],
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'selected_reference_baseline': 'soxl_backtest_policy_reimplementation_v1',
    'soxlbacktest_reference_role': 'qlib_reference_baseline_not_actor',
    'selected_diagnostic_candidate': 'official_replay::short_volshock_cap05_hit2_no_guard',
    'selected_diagnostic_candidate_reason': 'closest official replay near miss but still not retention eligible',
    'training_allowed_next': True,
    'long_ppo_allowed': False,
    'next_mechanism': 'fold_local_drawdown_drift_reward_or_checkpoint_screen_with_full_wf2020_wf2022_wf2023_2024_gate',
    'blocked_paths': [
        'retain any actor without strict walk-forward profile pass',
        'retain SOXLBacktest reference as actor',
        'extend reward_ppo_selected despite wf_2022 drawdown drift',
        'promote two-fold near miss without wf_2020 and fixed-split replay',
    ],
}

evidence_matrix = candidate_matrix[[
    'source_label',
    'candidate_id',
    'candidate_type',
    'windows_evaluated',
    'min_relative_return_pp',
    'max_dd_pct',
    'screen_gate_pass',
    'retention_gate_pass',
    'blocker',
]].copy()

source_manifest.to_csv(CARD / 'source_manifest.csv', index=False)
candidate_matrix.to_csv(CARD / 'candidate_gate_matrix.csv', index=False)
evidence_matrix.to_csv(CARD / 'evidence_matrix.csv', index=False)
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True) + '\n', encoding='utf-8')
(CARD / 'selection.json').write_text(json.dumps(selection, indent=2, sort_keys=True) + '\n', encoding='utf-8')

print(json.dumps({
    'decision': summary['decision'],
    'retention_gate_pass_count': retention_gate_pass_count,
    'candidate_row_count': len(candidate_matrix),
    'best_near_miss': best_near_miss['candidate_id'],
    'next_mechanism': selection['next_mechanism'],
}, indent=2))


{
  "decision": "short_foldlocal_drawdown_drift_no_retained_actor",
  "retention_gate_pass_count": 0,
  "candidate_row_count": 9,
  "best_near_miss": "official_replay::short_volshock_cap05_hit2_no_guard",
  "next_mechanism": "fold_local_drawdown_drift_reward_or_checkpoint_screen_with_full_wf2020_wf2022_wf2023_2024_gate"
}


## Conclusion

- No current actor or short-training candidate is retained. The strict profile gate has zero passing profiles and zero actor-retention-allowed candidates.
- `soxl_backtest_policy_reimplementation_v1` remains a Qlib-aligned buy/sell reference baseline only; it is not a runnable or promotable actor.
- The closest short-volshock official replay is still a near miss, and the reward PPO selected checkpoint fails drawdown-drift in `wf_2022`.
- Next work should be a fold-local drawdown-drift reward or checkpoint screen with full `wf_2020`, `wf_2022`, and `wf_2023_2024` gates before any actor retention.